In [9]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

# Читаем CSV в датафрейм (таблицу). Путь ../ — на уровень вверх из notebooks/ в data/
df = pd.read_csv("../data/paysim.csv")

df.shape   

(6362620, 11)

In [ ]:
df.head()  

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
df.info()  

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [ ]:
df.isnull().sum()   

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [ ]:
# Считаем метку isFraud 
print(df['isFraud'].value_counts())                
print(df['isFraud'].value_counts(normalize=True))  

isFraud
0    6354407
1       8213
Name: count, dtype: int64
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


In [ ]:
# Таблица 
pd.crosstab(df['type'], df['isFraud'], margins=True)

isFraud,0,1,All
type,,,
CASH_IN,1399284,0,1399284
CASH_OUT,2233384,4116,2237500
DEBIT,41432,0,41432
PAYMENT,2151495,0,2151495
TRANSFER,528812,4097,532909
All,6354407,8213,6362620


In [7]:
# Оставляем только типы операций, где встречается фрод
fraud_types = ['TRANSFER', 'CASH_OUT']
df_model = df[df['type'].isin(fraud_types)]

print("Было строк:      ", len(df))
print("Стало строк:     ", len(df_model))
print("Доля фрода стала:", df_model['isFraud'].mean())

Было строк:       6362620
Стало строк:      2770409
Доля фрода стала: 0.002964544224336551


In [8]:
import sys
sys.path.append("..")  
from scoring.features import add_instant_features

df_feat = add_instant_features(df_model)
df_feat[["amount", "errorBalanceOrig", "errorBalanceDest",
         "amountToBalanceRatio", "origBalanceZeroed", "hourOfDay"]].head()

,amount,errorBalanceOrig,errorBalanceDest,amountToBalanceRatio,origBalanceZeroed,hourOfDay
2,181.00,0.00,181.0,0.994505,1,1
3,181.00,0.00,21363.0,0.994505,1,1
15,229133.94,-213808.94,182703.5,14.950668,1,1
19,215310.30,-214605.30,237735.3,304.972096,1,1
24,311685.89,-300850.89,-2401220.0,28.763925,1,1


In [11]:
import importlib
import scoring.features
importlib.reload(scoring.features)          # принудительно перечитываем файл с диска
from scoring.features import add_velocity_features

df_vel = add_velocity_features(df_model)

print("Максимум операций у одного счёта (до текущей):", df_vel["txnCountSoFar"].max())
print(df_vel["txnCountSoFar"].value_counts().head())

Максимум операций у одного счёта (до текущей): 2
txnCountSoFar
0    2768630
1       1776
2          3
Name: count, dtype: int64


In [12]:
n = len(df)
print("Всего операций:      ", n)
print("Уникальных отправителей:", df["nameOrig"].nunique(), f"({df['nameOrig'].nunique()/n:.1%})")
print("Уникальных получателей: ", df["nameDest"].nunique(), f"({df['nameDest'].nunique()/n:.1%})")

Всего операций:       6362620
Уникальных отправителей: 6353307 (99.9%)
Уникальных получателей:  2722362 (42.8%)


In [13]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import add_velocity_features

# Считаем velocity на ВСЁМ df (вся история получателя, все типы операций)
df_vel = add_velocity_features(df)

print("Макс. входящих у одного получателя:", df_vel["destTxnCountSoFar"].max())
print(df_vel[["destTxnCountSoFar", "destAmountSoFar"]].describe())

Макс. входящих у одного получателя: 112
       destTxnCountSoFar  destAmountSoFar
count       6.362620e+06     6.362620e+06
mean        5.096119e+00     1.357718e+06
std         7.846729e+00     3.199564e+06
min         0.000000e+00     0.000000e+00
25%         0.000000e+00     0.000000e+00
50%         1.000000e+00     2.273446e+05
75%         7.000000e+00     1.552500e+06
max         1.120000e+02     3.572774e+08


In [14]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import build_features, FEATURE_COLUMNS

df_all = build_features(df)
print("Размер таблицы признаков:", df_all[FEATURE_COLUMNS].shape)
df_all[FEATURE_COLUMNS].head()

Размер таблицы признаков: (6362620, 8)


,amount,errorBalanceOrig,errorBalanceDest,amountToBalanceRatio,origBalanceZeroed,hourOfDay,destTxnCountSoFar,destAmountSoFar
0,9839.64,0.00,9839.64,0.057834,0,1,0,0.0
1801,5157.05,-3489.13,5157.05,3.090052,1,1,0,0.0
1802,5746.44,-5746.44,5746.44,5746.440000,1,1,0,0.0
1803,5607.36,-405.36,5607.36,1.077717,1,1,0,0.0
1804,6360.79,-2629.79,6360.79,1.704392,1,1,0,0.0


In [15]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import instant_features_one

txn = {"step": 1, "amount": 181.0,
       "oldbalanceOrg": 181.0, "newbalanceOrig": 0.0,
       "oldbalanceDest": 21182.0, "newbalanceDest": 0.0}

instant_features_one(txn)

{'amount': 181.0,
 'errorBalanceOrig': 0.0,
 'errorBalanceDest': 21363.0,
 'amountToBalanceRatio': 0.9945054945054945,
 'origBalanceZeroed': 1,
 'hourOfDay': 1}

In [16]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import VelocityState

state = VelocityState()
txns = [
    {"nameDest": "C999", "amount": 100.0},
    {"nameDest": "C999", "amount": 200.0},
    {"nameDest": "C888", "amount": 50.0},
    {"nameDest": "C999", "amount": 300.0},
]
for t in txns:
    print(t["nameDest"], t["amount"], "->", state.velocity_features_one(t))

C999 100.0 -> {'destTxnCountSoFar': 0, 'destAmountSoFar': 0.0}
C999 200.0 -> {'destTxnCountSoFar': 1, 'destAmountSoFar': 100.0}
C888 50.0 -> {'destTxnCountSoFar': 0, 'destAmountSoFar': 0.0}
C999 300.0 -> {'destTxnCountSoFar': 2, 'destAmountSoFar': 300.0}


In [17]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import VelocityState, FEATURE_COLUMNS

state = VelocityState()
txn = {"step": 1, "amount": 181.0,
       "oldbalanceOrg": 181.0, "newbalanceOrig": 0.0,
       "nameDest": "C553264065", "oldbalanceDest": 0.0, "newbalanceDest": 0.0}

feats = state.features_one(txn)
print(feats)
print("Набор совпадает с FEATURE_COLUMNS:", set(feats.keys()) == set(FEATURE_COLUMNS))

{'amount': 181.0, 'errorBalanceOrig': 0.0, 'errorBalanceDest': 181.0, 'amountToBalanceRatio': 0.9945054945054945, 'origBalanceZeroed': 1, 'hourOfDay': 1, 'destTxnCountSoFar': 0, 'destAmountSoFar': 0.0}
Набор совпадает с FEATURE_COLUMNS: True


In [18]:
import joblib, pandas as pd
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import VelocityState, FEATURE_COLUMNS

# Загружаем обученную модель (путь из notebooks/ на уровень вверх в models/)
model = joblib.load("../models/fraud_model.pkl")
state = VelocityState()

# Одна "живая" транзакция: перевод, счёт отправителя обнулён под ноль (похоже на фрод)
txn = {"step": 5, "amount": 181.0,
       "oldbalanceOrg": 181.0, "newbalanceOrig": 0.0,
       "nameDest": "C553264065", "oldbalanceDest": 0.0, "newbalanceDest": 0.0}

# 1. Считаем 8 признаков потоковым путём
feats = state.features_one(txn)

# 2. Приводим к формату модели: одна строка, колонки в ПРАВИЛЬНОМ порядке
X = pd.DataFrame([feats])[FEATURE_COLUMNS]

# 3. Спрашиваем модель: вероятность, что это фрод
prob = model.predict_proba(X)[0, 1]
print("Вероятность фрода:", round(prob, 4))

Вероятность фрода: 1.0


In [19]:
normal = {"step": 5, "amount": 500.0,
          "oldbalanceOrg": 20000.0, "newbalanceOrig": 19500.0,   # 20000 - 500 = 19500, сходится
          "nameDest": "C111", "oldbalanceDest": 1000.0, "newbalanceDest": 1500.0}  # 1000 + 500 = 1500, сходится

feats_n = state.features_one(normal)
X_n = pd.DataFrame([feats_n])[FEATURE_COLUMNS]
print("Вероятность фрода (честная операция):", round(model.predict_proba(X_n)[0, 1], 4))

Вероятность фрода (честная операция): 0.0


In [20]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import RedisVelocityState

state = RedisVelocityState()
state.r.flushdb()   # очищаем Redis перед тестом — чистый старт (сотрёт все ключи)

txns = [
    {"nameDest": "C999", "amount": 100.0},
    {"nameDest": "C999", "amount": 200.0},
    {"nameDest": "C888", "amount": 50.0},
    {"nameDest": "C999", "amount": 300.0},
]
for t in txns:
    print(t["nameDest"], t["amount"], "->", state.velocity_features_one(t))

C999 100.0 -> {'destTxnCountSoFar': 0, 'destAmountSoFar': 0.0}
C999 200.0 -> {'destTxnCountSoFar': 1, 'destAmountSoFar': 100.0}
C888 50.0 -> {'destTxnCountSoFar': 0, 'destAmountSoFar': 0.0}
C999 300.0 -> {'destTxnCountSoFar': 2, 'destAmountSoFar': 300.0}


In [21]:
import importlib, scoring.features
importlib.reload(scoring.features)
from scoring.features import RedisVelocityState

state = RedisVelocityState() 

txns = [
    {"nameDest": "C999", "amount": 100.0},
    {"nameDest": "C999", "amount": 200.0},
    {"nameDest": "C888", "amount": 50.0},
    {"nameDest": "C999", "amount": 300.0},
]
for t in txns:
    print(t["nameDest"], t["amount"], "->", state.velocity_features_one(t))

C999 100.0 -> {'destTxnCountSoFar': 3, 'destAmountSoFar': 600.0}
C999 200.0 -> {'destTxnCountSoFar': 4, 'destAmountSoFar': 700.0}
C888 50.0 -> {'destTxnCountSoFar': 1, 'destAmountSoFar': 50.0}
C999 300.0 -> {'destTxnCountSoFar': 5, 'destAmountSoFar': 900.0}


In [22]:
import importlib, scoring.features, scoring.rules
importlib.reload(scoring.features); importlib.reload(scoring.rules)
from scoring.features import RedisVelocityState
from scoring.rules import check_rules

state = RedisVelocityState()
state.r.flushdb()

fraud  = {"step": 5, "amount": 181.0, "oldbalanceOrg": 181.0, "newbalanceOrig": 0.0,
          "nameDest": "C553264065", "oldbalanceDest": 0.0, "newbalanceDest": 0.0}
normal = {"step": 5, "amount": 500.0, "oldbalanceOrg": 20000.0, "newbalanceOrig": 19500.0,
          "nameDest": "C111", "oldbalanceDest": 1000.0, "newbalanceDest": 1500.0}

print("fraud :", check_rules(fraud,  state.features_one(fraud)))
print("normal:", check_rules(normal, state.features_one(normal)))

fraud : ['orig_balance_zeroed']
normal: []


In [23]:
import joblib, pandas as pd, importlib
import scoring.features, scoring.rules, scoring.decision
importlib.reload(scoring.features); importlib.reload(scoring.rules); importlib.reload(scoring.decision)
from scoring.features import RedisVelocityState, FEATURE_COLUMNS
from scoring.rules import check_rules
from scoring.decision import decide

model = joblib.load("../models/fraud_model.pkl")
state = RedisVelocityState(); state.r.flushdb()

def score(txn):
    feats = state.features_one(txn)                                    # 1. признаки
    prob  = model.predict_proba(pd.DataFrame([feats])[FEATURE_COLUMNS])[0, 1]  # 2. ML-скор
    rules = check_rules(txn, feats)                                    # 3. правила
    return decide(prob, rules), round(prob, 3), rules                  # 4. решение

fraud  = {"step": 5, "amount": 181.0, "oldbalanceOrg": 181.0, "newbalanceOrig": 0.0,
          "nameDest": "C553264065", "oldbalanceDest": 0.0, "newbalanceDest": 0.0}
normal = {"step": 5, "amount": 500.0, "oldbalanceOrg": 20000.0, "newbalanceOrig": 19500.0,
          "nameDest": "C111", "oldbalanceDest": 1000.0, "newbalanceDest": 1500.0}

print("fraud :", score(fraud))
print("normal:", score(normal))

fraud : ('block', np.float64(1.0), ['orig_balance_zeroed'])
normal: ('allow', np.float64(0.0), [])


In [24]:
import importlib, scoring.store
importlib.reload(scoring.store)
from scoring.store import get_connection, save_decision

conn = get_connection("../db/fraud.db")   # путь из notebooks/ на уровень вверх

# сохраняем решение по fraud-операции (добавим nameOrig, которого не было в тесте)
save_decision(conn,
              {**fraud, "nameOrig": "C001", "isFraud": 1},
              prob=1.0, fired_rules=["orig_balance_zeroed"], decision="block")

# читаем обратно всё, что записано
print("Записи в базе:")
for row in conn.execute("SELECT step, amount, decision, fired_rules FROM decisions"):
    print(row)
conn.close()

Записи в базе:
(5, 181.0, 'block', 'orig_balance_zeroed')
